# Notebook 09 — Polynomial Addition over GF(2)
**Mathematics of Cryptography · Volume 1**

In Module 08 we saw that every byte is a polynomial whose coefficients are 0 or 1.  
This notebook explores what *adding* two such polynomials means — and why it is exactly the same as XOR.

**Sections**
1. GF(2) coefficient arithmetic  
2. Polynomial addition = XOR  
3. Visualising term cancellation  
4. Verifying with 0x57 ⊕ 0x83  
5. Self-XOR: every byte ⊕ itself = 0  
6. Commutativity and associativity  
7. Why integer addition is wrong here  
8. Interactive XOR explorer  
9. AES connection: the AddRoundKey layer  
10. Summary and bridge to Module 10

---
## Section 1 — GF(2) Coefficient Arithmetic

GF(2) has only two values: **0** and **1**.  
Addition is modulo 2 — the only surprise is that 1 + 1 = 0.

In [ ]:
def gf2_add(a: int, b: int) -> int:
    """Add two GF(2) coefficients (each 0 or 1). Result is (a + b) mod 2."""
    assert a in (0, 1) and b in (0, 1), "coefficients must be 0 or 1"
    return (a + b) % 2

# Print the complete GF(2) addition table
print("GF(2) addition table")
print("-" * 30)
print(f"{'A':>4} {'B':>4} {'A+B (ord)':>10} {'A+B mod 2':>10} {'A XOR B':>8}")
print("-" * 30)
for a in (0, 1):
    for b in (0, 1):
        ordinary = a + b
        gf2     = gf2_add(a, b)
        xor_val  = a ^ b
        match = "✓" if gf2 == xor_val else "✗"
        print(f"  {a:>2}   {b:>2}   {ordinary:>8}   {gf2:>8}   {xor_val:>6}  {match}")

print()
print("Key observation: GF(2) addition and XOR produce identical results.")

---
## Section 2 — Polynomial Addition = XOR

To add two byte-polynomials, add their coefficients at every power of x — each coefficient pair independently, modulo 2.  
For 8-bit bytes this is equivalent to XORing the two integers.

In [ ]:
def poly_add(a: int, b: int) -> int:
    """Add two byte-polynomials over GF(2). Returns a single byte integer."""
    return a ^ b

def byte_to_poly(b: int) -> str:
    """Return a human-readable polynomial string for a byte value."""
    if b == 0:
        return "0"
    terms = []
    for k in range(7, -1, -1):
        if b & (1 << k):
            if k == 0:
                terms.append("1")
            elif k == 1:
                terms.append("x")
            else:
                terms.append(f"x^{k}")
    return " + ".join(terms)

# Demonstrate
pairs = [(0x57, 0x83), (0x1B, 0x03), (0xFF, 0x0F), (0xAA, 0x55), (0x63, 0x63)]

print(f"{'A':>6}  {'B':>6}  {'A+B':>6}  Polynomial result")
print("-" * 60)
for a, b in pairs:
    result = poly_add(a, b)
    print(f"  0x{a:02X}  0x{b:02X}  0x{result:02X}  {byte_to_poly(result)}")

# Verify poly_add gives same result as plain XOR
print()
all_match = all(poly_add(a, b) == (a ^ b) for a in range(256) for b in range(256))
print(f"poly_add(a,b) == a ^ b for all 65536 pairs: {all_match}")

---
## Section 3 — Visualising Term Cancellation

When both polynomials share a term, the coefficient sum is 1+1 = 0 and the term **cancels**.  
When only one polynomial has the term, the coefficient sum is 1+0 = 1 and the term **survives**.

In [ ]:
def cancellation_table(a: int, b: int) -> None:
    """Print a term-by-term cancellation breakdown for two bytes."""
    result = a ^ b
    print(f"  A = 0x{a:02X} = {a:08b}  =  {byte_to_poly(a)}")
    print(f"  B = 0x{b:02X} = {b:08b}  =  {byte_to_poly(b)}")
    print()
    print(f"  {'Power':>6}  {'Bit A':>5}  {'Bit B':>5}  {'1+1 mod2':>8}  {'Fate':>10}")
    print("  " + "-" * 44)
    for k in range(7, -1, -1):
        ca = (a >> k) & 1
        cb = (b >> k) & 1
        s  = ca ^ cb
        term = f"x^{k}" if k > 1 else ("x" if k == 1 else "1")
        fate = "CANCELS" if (ca == 1 and cb == 1) else ("survives" if s == 1 else "absent")
        print(f"  {term:>6}     {ca:>3}     {cb:>3}     {s:>5}       {fate}")
    print()
    print(f"  Result = 0x{result:02X} = {byte_to_poly(result)}")

print("=" * 50)
print("Example: 0x57 + 0x83")
print("=" * 50)
cancellation_table(0x57, 0x83)

print()
print("=" * 50)
print("Example: 0x1B + 0x03")
print("=" * 50)
cancellation_table(0x1B, 0x03)

---
## Section 4 — Worked Example: 0x57 ⊕ 0x83

The canonical AES example from the tutorial. We verify all three representations agree.

In [ ]:
A = 0x57  # x^6 + x^4 + x^2 + x + 1
B = 0x83  # x^7 + x + 1

# Method 1: XOR the integers
xor_result = A ^ B

# Method 2: polynomial addition term-by-term
poly_result = poly_add(A, B)

# Method 3: hand-checked expected value from the tutorial
expected = 0xD4  # x^7 + x^6 + x^4 + x^2

print(f"A      = 0x{A:02X}  ({A:08b})  {byte_to_poly(A)}")
print(f"B      = 0x{B:02X}  ({B:08b})  {byte_to_poly(B)}")
print(f"A XOR B= 0x{xor_result:02X}  ({xor_result:08b})  {byte_to_poly(xor_result)}")
print()
print(f"All three methods agree: {xor_result == poly_result == expected}")
print(f"Expected 0x{expected:02X}: {xor_result == expected}")

---
## Section 5 — Self-XOR: A ⊕ A = 0 for Every Byte

Adding any polynomial to itself means every coefficient becomes 1+1 = 0.  
So the result is always the zero polynomial.

In [ ]:
# Verify: a XOR a == 0 for all 256 byte values
failures = [a for a in range(256) if poly_add(a, a) != 0]
print(f"A ⊕ A = 0 holds for all 256 bytes: {len(failures) == 0}")

# Show a few examples
print()
print("Sample self-additions:")
for val in [0x57, 0x83, 0x1B, 0xFF, 0x01]:
    result = poly_add(val, val)
    print(f"  0x{val:02X} ⊕ 0x{val:02X} = 0x{result:02X}  ({'zero polynomial ✓' if result == 0 else 'ERROR'})")

print()
print("Consequence: XOR is its own inverse. A ⊕ B ⊕ B = A (used in stream cipher decryption).")

---
## Section 6 — Commutativity and Associativity

Polynomial addition over GF(2) inherits the algebraic properties of XOR.

In [ ]:
import random
random.seed(42)
samples = [(random.randint(0,255), random.randint(0,255), random.randint(0,255))
           for _ in range(10000)]

# Commutativity: a + b == b + a
comm_ok = all(poly_add(a, b) == poly_add(b, a) for a, b, _ in samples)

# Associativity: (a + b) + c == a + (b + c)
assoc_ok = all(
    poly_add(poly_add(a, b), c) == poly_add(a, poly_add(b, c))
    for a, b, c in samples
)

# Identity: a + 0 == a
ident_ok = all(poly_add(a, 0) == a for a in range(256))

# Self-inverse: a + a == 0
inv_ok = all(poly_add(a, a) == 0 for a in range(256))

print("Algebraic properties of GF(2) polynomial addition:")
print(f"  Commutative  (a+b = b+a):           {comm_ok}")
print(f"  Associative  ((a+b)+c = a+(b+c)):   {assoc_ok}")
print(f"  Identity     (a+0 = a):             {ident_ok}")
print(f"  Self-inverse (a+a = 0):             {inv_ok}")
print()
print("These four properties define an abelian group — the foundation for field arithmetic.")

---
## Section 7 — Why Integer Addition Is Wrong Here

A common mistake is treating byte values as integers and adding them ordinarily.  
This section shows how integer addition and GF(2) addition diverge.

In [ ]:
print("Comparing GF(2) addition vs integer addition for selected pairs:\n")
print(f"  {'A':>5}  {'B':>5}  {'Int A+B':>8}  {'GF(2) A+B':>11}  {'Same?':>6}")
print("  " + "-" * 44)

test_pairs = [(0x57, 0x83), (0x01, 0x01), (0xFF, 0xFF), (0x1B, 0x1B), (0x0F, 0xF0), (0x57, 0x57)]
for a, b in test_pairs:
    int_sum = a + b           # ordinary integer addition
    gf2_sum = poly_add(a, b)  # GF(2) polynomial addition
    same = "YES" if int_sum == gf2_sum else "NO"
    print(f"  0x{a:02X}  0x{b:02X}  0x{int_sum:04X} ({int_sum:>4})  0x{gf2_sum:02X} ({gf2_sum:>3})  {same:>6}")

print()
print("Key: integer addition can overflow 255, carries between bits, and never gives 0 unless one operand is 0.")
print("GF(2) addition wraps each bit independently — no carry, no overflow.")

---
## Section 8 — Interactive XOR Explorer

Given any two hex byte values, this function breaks down the XOR into individual bit positions,  
showing exactly which terms cancel and which survive.

In [ ]:
def xor_explorer(a_hex: str, b_hex: str) -> None:
    """
    Interactive XOR breakdown.
    Pass hex strings like '0x57' or '57'.
    """
    a = int(a_hex, 16)
    b = int(b_hex, 16)
    assert 0 <= a <= 255 and 0 <= b <= 255, "values must be single bytes (0–255)"

    result = a ^ b

    print(f"━━━ XOR Explorer: 0x{a:02X} + 0x{b:02X} ━━━")
    print(f"  A = 0x{a:02X}  binary: {a:08b}  polynomial: {byte_to_poly(a)}")
    print(f"  B = 0x{b:02X}  binary: {b:08b}  polynomial: {byte_to_poly(b)}")
    print()
    print(f"  {'Power':>6}  cA  cB  cA⊕cB  Outcome")
    print("  " + "-" * 38)
    for k in range(7, -1, -1):
        ca = (a >> k) & 1
        cb = (b >> k) & 1
        s  = ca ^ cb
        term = f"x^{k}" if k > 1 else ("x  " if k == 1 else "1  ")
        if ca == 1 and cb == 1:
            outcome = "↯ cancel"
        elif s == 1:
            outcome = "✓ survive"
        else:
            outcome = "  absent"
        print(f"  {term:>6}   {ca}   {cb}    {s}     {outcome}")
    print()
    print(f"  Result = 0x{result:02X}  binary: {result:08b}  polynomial: {byte_to_poly(result)}")

# ── Try different pairs here ──────────────────────────────────────────
xor_explorer('0x57', '0x83')
print()
xor_explorer('0x1B', '0x63')

---
## Section 9 — AES Connection: The AddRoundKey Layer

In AES, every round begins with **AddRoundKey**: each byte of the 4×4 state matrix is XORed  
with the corresponding byte of the round key.  
This is exactly GF(2) polynomial addition — one byte-polynomial at a time.

In [ ]:
def add_round_key(state: list[list[int]], key: list[list[int]]) -> list[list[int]]:
    """XOR each byte of a 4x4 state with the corresponding key byte."""
    return [[state[r][c] ^ key[r][c] for c in range(4)] for r in range(4)]

# Toy example: a small 4x4 state and round key
state = [
    [0x32, 0x88, 0x31, 0xE0],
    [0x43, 0x5A, 0x31, 0x37],
    [0xF6, 0x30, 0x98, 0x07],
    [0xA8, 0x8D, 0xA2, 0x34],
]

round_key = [
    [0x2B, 0x28, 0xAB, 0x09],
    [0x7E, 0xAE, 0xF7, 0xCF],
    [0x15, 0xD2, 0x15, 0x4F],
    [0x16, 0xA6, 0x88, 0x3C],
]

result_state = add_round_key(state, round_key)

print("AddRoundKey: state ⊕ round_key")
print()
for r in range(4):
    s_row  = "  ".join(f"0x{b:02X}" for b in state[r])
    k_row  = "  ".join(f"0x{b:02X}" for b in round_key[r])
    rs_row = "  ".join(f"0x{b:02X}" for b in result_state[r])
    print(f"  [{s_row}]  ⊕  [{k_row}]  =  [{rs_row}]")

print()
print("Each ⊕ is one GF(2) polynomial addition — exactly what Module 09 teaches.")
print()
# Verify decryption reversal: applying the same key again recovers the state
recovered = add_round_key(result_state, round_key)
print(f"Applying the round key a second time recovers the original state: {recovered == state}")
print("(This works because A ⊕ K ⊕ K = A — the self-inverse property.)")

---
## Section 10 — Summary and Bridge to Module 10

| Concept | Key fact |
|---|---|
| GF(2) coefficient addition | `(a + b) mod 2` — only 0 and 1 possible |
| Duplicate terms | Cancel: `1 + 1 = 0` |
| Polynomial addition | XOR all 8 bit positions independently |
| No carrying | Each bit position is self-contained |
| Subtraction | Identical to addition in GF(2) |
| Self-inverse | `A ⊕ A = 0` for every byte |
| AES use | `AddRoundKey` layer is 16 independent GF(2) additions |

In [ ]:
print("Module 09 in one function:")
print()
print("  poly_add(a, b) = a ^ b")
print()
print("That is the complete implementation of GF(2) polynomial addition.")
print()
print("What Module 10 adds:")
print("  poly_mul(a, b) = ?")
print()
print("Multiplication creates new powers of x — terms like x^8 can appear.")
print("Those must be reduced back to degree < 8 using the AES irreducible polynomial.")
print("That reduction is the core of Module 11 and the heart of GF(2^8) arithmetic.")